# Project-Wide Evaluation-Metrics Audit, Standardization and Descriptive Backfill

Scope: the completed KUB research chain (Phases 09–14 and the post-Phase14 H3
training / supply / downstream steps) up to the state
`H3_DOWNSTREAM_EVALUATION_COMPLETE__FORMAL_ADJUDICATION_PENDING`.

This notebook is an **audit + standardization** record. It performs **no training,
no inference and no TEST access**, and it does not change any frozen gate.

**Terminal state:** `PROJECT_EVALUATION_METRICS_AUDIT_COMPLETE__H3_ADJUDICATION_PENDING`

## 1. Scope

In [1]:
import json, hashlib, sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

ROOT = Path('D:/Softfors/jup')
sys.path.insert(0, str(ROOT))
AUDITS = ROOT / 'reports' / 'audits'
TABLES = ROOT / 'reports' / 'tables'
FIGDIR = ROOT / 'reports' / 'figures'
FIGDIR.mkdir(parents=True, exist_ok=True)

def load_json(p): return json.loads((ROOT / p).read_text(encoding='utf-8'))
def sha(p): return hashlib.sha256((ROOT / p).read_bytes()).hexdigest()

inv = load_json('reports/audits/project_evaluation_metrics_inventory.json')
policy = load_json('reports/audits/project_evaluation_metric_policy.json')
empty_pol = load_json('reports/audits/evaluation_empty_mask_policy.json')
registry = load_json('reports/audits/project_metric_registry.json')
backfill = load_json('reports/audits/project_evaluation_metrics_backfill.json')
proposal = load_json('reports/audits/project_evaluation_proposal_metrics_backfill.json')

summary_df = pd.read_csv(TABLES / 'project_evaluation_segmentation_summary.csv')
perimg_df = pd.read_csv(TABLES / 'project_evaluation_per_image_metrics.csv')
size_df = pd.read_csv(TABLES / 'project_evaluation_component_recall_by_size.csv')
cov_df = pd.read_csv(TABLES / 'project_evaluation_coverage_sensitivity.csv')
paired_df = pd.read_csv(TABLES / 'project_evaluation_paired_comparisons.csv')
conf = load_json('reports/tables/project_evaluation_confusion_matrices.json')
prop_summary = pd.read_csv(TABLES / 'project_evaluation_proposal_metrics_summary.csv')
curves = load_json('reports/tables/project_evaluation_proposal_curves.json')

print('Audited scope:', inv['scope'])
print('Inventory rows:', len(inv['rows']))
print('Registry metrics:', len(registry['metrics']))
print('Backfill status:', backfill['status'])
print('Proposal backfill status:', proposal['status'])
print('TEST reads:', backfill['governance']['test_reads'], '| external:', backfill['governance']['external_test_reads'])

Audited scope: KUB research line: Phases 09-14, post-Phase14 H3 (training / supply / downstream), and this audit.
Inventory rows: 12
Registry metrics: 35
Backfill status: PROJECT_EVALUATION_METRICS_BACKFILL_COMPLETE__DESCRIPTIVE_ONLY
Proposal backfill status: PROPOSAL_DESCRIPTIVE_METRICS_COMPLETE__NO_INFERENCE
TEST reads: 0 | external: 0


## 2. Metric-Domain Taxonomy

In [2]:
display(pd.DataFrame(list(inv['metric_domains'].items()), columns=['code', 'metric_domain']))
print()
print('Policy label rule:', policy['label_rule'])
print('Policy distance-unit rule:', policy['distance_unit_rule'])

,code,metric_domain
0,A,PIXEL_SEGMENTATION
1,B,OBJECT_COMPONENT_DETECTION
2,C,LOCALIZATION_PROPOSAL_SUPPLY
3,D,BOUNDARY_SURFACE_QUALITY
4,E,PROBABILITY_CALIBRATION
5,F,TRAINING_OPTIMIZATION
6,G,CAUSAL_COUNTERFACTUAL_DIAGNOSTIC
7,H,SYSTEM_PIPELINE_PERFORMANCE



Policy label rule: Never use the same short label (e.g. 'Recall') across domains without explicit qualification.
Policy distance-unit rule: No pixel-spacing metadata exists in this repository; boundary distances are reported in PIXELS, never mm.


## 3. Confusion-Matrix Policy

In [3]:
rows = []
for task, d in policy['policies'].items():
    rows.append({'task_type': task, 'confusion_matrix': d['confusion_matrix'],
                 'form': d['form'], 'reason': d.get('reason', '')})
display(pd.DataFrame(rows))
print()
print('Empty-mask policy principle:', empty_pol['principle'])
display(pd.DataFrame([{'case': k, 'dice': v.get('dice'), 'recall': v.get('recall'),
                       'specificity': v.get('specificity'), 'hd95': v.get('hd95')}
                      for k, v in empty_pol['cases'].items()]))

,task_type,confusion_matrix,form,reason
0,segmentation,YES,2x2 pixel confusion matrix (Background / Foreg...,
1,image_level_binary_classification,YES,standard 2x2 confusion matrix,
2,component_detection,NO_CONVENTIONAL_TN,TP / FP / FN summary + PR / FROC,conventional TN is undefined without a defensi...
3,proposal_localization,NO_FORCED_TN,matched / unmatched proposal and GT accounting,
4,multi_class_segmentation_future,YES,CxC confusion matrix + per-class metrics,



Empty-mask policy principle: Undefined metrics return NaN together with an explicit validity flag. They are NEVER silently zeroed.


,case,dice,recall,specificity,hd95
0,gt_empty_pred_empty,1.0,1.0,1.0,NaN (invalid: both_empty)
1,gt_empty_pred_nonempty,0.0,NaN (undefined: no positives),1.0,NaN (invalid)
2,gt_nonempty_pred_empty,0.0,0.0,1.0,NaN (invalid: empty_prediction)
3,both_nonempty,NaN,None,NaN,NaN


## 4. Current Phase / Step Inventory

In [4]:
inv_df = pd.DataFrame(inv['rows'])
display(inv_df[['phase_or_step', 'task_type', 'metric_domains', 'requires_new_inference', 'status']])

,phase_or_step,task_type,metric_domains,requires_new_inference,status
0,Phase 09 — KUB UNet baseline training,TRAINING,[F],True,EXISTING_PARTIAL__TRAINING_SCOPE_METRICS_SUFFI...
1,Phase 10 — baseline comparison (clean-v3),SEGMENTATION,"[A, H]",True,EXISTING_PARTIAL__DESCRIPTIVE_BACKFILL_AVAILABLE
2,Phase 11 — failure analysis,DIAGNOSTIC,[G],False,EXISTING_SUFFICIENT
3,Phase 12 — oracle ROI,LOCALIZATION,[C],False,EXISTING_SUFFICIENT__ORACLE_BOUND
4,Phase 13 — predicted ROI / localizer,LOCALIZATION,"[C, B]",False,EXISTING_PARTIAL__PR_FROC_DERIVABLE
5,Phase 13 Part1D — predicted-region downstream,SEGMENTATION,"[A, B]",True,EXISTING_PARTIAL__PREDICTIONS_NOT_PERSISTED
6,Phase 14 — regional refiner diagnostic (clean-v3),SEGMENTATION+DETECTION,"[A, B, H]",True,EXISTING_PARTIAL__BACKFILLED_DESCRIPTIVELY
7,Post-Phase14 root-cause analysis,CAUSAL_DIAGNOSTIC,[G],False,EXISTING_SUFFICIENT
8,H3 training (size-adaptive sigma localizer),TRAINING,"[F, C]",False,EXISTING_SUFFICIENT__TRAINING_SCOPE
9,H3 supply-axis evaluation,LOCALIZATION,"[C, B]",False,EXISTING_PARTIAL__PROPOSAL_DOMAIN_ONLY


## 5. Existing-vs-Missing Metric Matrix

In [5]:
inv_csv = pd.read_csv(TABLES / 'project_evaluation_metrics_inventory.csv')
display(inv_csv[['phase_or_step', 'existing_metrics', 'missing_recommended_metrics', 'status']])

,phase_or_step,existing_metrics,missing_recommended_metrics,status
0,Phase 09 — KUB UNet baseline training,train loss (bce_dice); val loss; val macro Dic...,per-image Dice distribution; IoU; pixel confus...,EXISTING_PARTIAL__TRAINING_SCOPE_METRICS_SUFFI...
1,Phase 10 — baseline comparison (clean-v3),mean/median Dice; macro Dice; runtime,IoU; precision; recall; specificity; MCC; pixe...,EXISTING_PARTIAL__DESCRIPTIVE_BACKFILL_AVAILABLE
2,Phase 11 — failure analysis,failure counts by category; empty-prediction a...,none required (qualitative/diagnostic),EXISTING_SUFFICIENT
3,Phase 12 — oracle ROI,oracle ROI coverage; retention metrics,proposal precision/recall; ROI area fraction,EXISTING_SUFFICIENT__ORACLE_BOUND
4,Phase 13 — predicted ROI / localizer,coverage_fraction; covered90; covered50; any_o...,PR curve; FROC; proposal-to-GT IoU; center dis...,EXISTING_PARTIAL__PR_FROC_DERIVABLE
5,Phase 13 Part1D — predicted-region downstream,per-model Dice; component coverage,pixel confusion; boundary metrics; calibration,EXISTING_PARTIAL__PREDICTIONS_NOT_PERSISTED
6,Phase 14 — regional refiner diagnostic (clean-v3),mean/median Dice; detection recall@90 by size;...,pixel confusion; IoU; precision; specificity; ...,EXISTING_PARTIAL__BACKFILLED_DESCRIPTIVELY
7,Post-Phase14 root-cause analysis,R_A; R_C; axis-priority verdict; component-lev...,none required (causal diagnostic),EXISTING_SUFFICIENT
8,H3 training (size-adaptive sigma localizer),val_localization_recall; val_loss; epoch timin...,PR/FROC on peak scores,EXISTING_SUFFICIENT__TRAINING_SCOPE
9,H3 supply-axis evaluation,zero-proposal GT/TINY; any-overlap recall; cov...,PR curve; FROC; proposal-to-GT IoU; center dis...,EXISTING_PARTIAL__PROPOSAL_DOMAIN_ONLY


## 6. Derivability Matrix

In [6]:
der = inv_csv[['phase_or_step', 'derivable_from_saved_artifacts', 'requires_new_inference',
               'scientifically_appropriate', 'usable_for_formal_adjudication']].copy()
display(der)
print()
print('Key finding: no per-pixel masks or probability maps were persisted by the historical')
print('evaluators, so pixel confusion / boundary / calibration metrics were NOT derivable')
print('from saved artifacts. They were obtained through ONE logged safe VAL replay.')
print('Replay verdict:', backfill['safe_replay_rule']['verdict'])

,phase_or_step,derivable_from_saved_artifacts,requires_new_inference,scientifically_appropriate,usable_for_formal_adjudication
0,Phase 09 — KUB UNet baseline training,per-image Dice (only aggregate persisted),True,True,False
1,Phase 10 — baseline comparison (clean-v3),per-image Dice (persisted for the GLOBAL arm),True,True,False
2,Phase 11 — failure analysis,none,False,True,False
3,Phase 12 — oracle ROI,coverage tables,False,True,False
4,Phase 13 — predicted ROI / localizer,coverage + best_proposal_score (enables descri...,False,True,False
5,Phase 13 Part1D — predicted-region downstream,none,True,True,False
6,Phase 14 — regional refiner diagnostic (clean-v3),per-image Dice; per-component overlap pixels; ...,True,True,False
7,Post-Phase14 root-cause analysis,none,False,True,False
8,H3 training (size-adaptive sigma localizer),epoch-level metrics table,False,True,False
9,H3 supply-axis evaluation,coverage + score tables,False,True,False



Key finding: no per-pixel masks or probability maps were persisted by the historical
evaluators, so pixel confusion / boundary / calibration metrics were NOT derivable
from saved artifacts. They were obtained through ONE logged safe VAL replay.
Replay verdict: REPLAY_PERFORMED_LAWFULLY


## 7. Segmentation Metrics

In [7]:
cols = ['arm', 'pooled_dice', 'pooled_iou', 'pooled_precision', 'pooled_recall', 'pooled_specificity',
        'pooled_npv', 'pooled_accuracy', 'pooled_balanced_accuracy', 'pooled_mcc', 'pooled_cohen_kappa',
        'pooled_tp', 'pooled_tn', 'pooled_fp', 'pooled_fn']
display(summary_df[[c for c in cols if c in summary_df.columns]])
print()
print('Image-level (mean / median / std / IQR / min / max / 95% bootstrap CI) for Dice:')
dcols = ['arm', 'image_mean_dice', 'image_median_dice', 'image_std_dice', 'image_iqr_dice',
         'image_min_dice', 'image_max_dice', 'dice_ci_low', 'dice_ci_high']
display(summary_df[[c for c in dcols if c in summary_df.columns]])
print()
print('CAUTION: under heavy class imbalance accuracy/specificity alone are not evidence of quality.')
print('Positive (foreground) pixel fraction per arm:')
display(summary_df[['arm', 'positive_fraction']])

,arm,pooled_dice,pooled_iou,pooled_precision,pooled_recall,pooled_specificity,pooled_npv,pooled_accuracy,pooled_balanced_accuracy,pooled_mcc,pooled_cohen_kappa,pooled_tp,pooled_tn,pooled_fp,pooled_fn
0,GLOBAL,0.665747,0.498966,0.790486,0.575010,0.999883,0.999675,0.999558,0.787446,0.673986,0.665531,358396,814085530,94991,264891
1,ORACLE_REGIONAL,0.751746,0.602238,0.821843,0.692666,0.999885,0.999765,0.999650,0.846276,0.754325,0.751572,431730,814086932,93589,191557
2,PREDICTED_REGIONAL,0.514746,0.346571,0.472663,0.565056,0.999517,0.999667,0.999185,0.782287,0.516396,0.514342,352192,813787590,392931,271095
3,H3_DOWNSTREAM,0.464786,0.302750,0.399027,0.556496,0.999358,0.999660,0.999020,0.777927,0.470757,0.464309,346857,813658120,522401,276430



Image-level (mean / median / std / IQR / min / max / 95% bootstrap CI) for Dice:


,arm,image_mean_dice,image_median_dice,image_std_dice,image_iqr_dice,image_min_dice,image_max_dice,dice_ci_low,dice_ci_high
0,GLOBAL,0.635917,0.745656,0.277271,0.252315,9.424182e-12,0.922579,0.581571,0.686591
1,ORACLE_REGIONAL,0.802416,0.834329,0.131780,0.116459,1.980198e-01,0.934107,0.776504,0.825454
2,PREDICTED_REGIONAL,0.418531,0.370672,0.234849,0.347741,7.085164e-12,0.861038,0.373291,0.463137
3,H3_DOWNSTREAM,0.388628,0.369360,0.215954,0.309829,7.629511e-12,0.836344,0.346645,0.429068



CAUTION: under heavy class imbalance accuracy/specificity alone are not evidence of quality.
Positive (foreground) pixel fraction per arm:


,arm,positive_fraction
0,GLOBAL,0.000765
1,ORACLE_REGIONAL,0.000765
2,PREDICTED_REGIONAL,0.000765
3,H3_DOWNSTREAM,0.000765


### Figure 1 — Raw pixel confusion matrices (counts)

In [8]:
arms = list(conf.keys())
fig, axes = plt.subplots(1, len(arms), figsize=(4.2 * len(arms), 4.2))
if len(arms) == 1: axes = [axes]
for ax, a in zip(axes, arms):
    m = np.array(conf[a]['matrix_raw'], dtype=float)
    ax.imshow(np.log10(m + 1), cmap='Blues')
    for i in range(2):
        for j in range(2):
            ax.text(j, i, f'{int(m[i, j]):,}', ha='center', va='center', fontsize=11,
                    color='black' if m[i, j] < m.max() * 0.6 else 'white')
    ax.set_xticks([0, 1]); ax.set_xticklabels(['Pred BG', 'Pred FG'])
    ax.set_yticks([0, 1]); ax.set_yticklabels(['GT BG', 'GT FG'])
    ax.set_title(f'{a}\n(raw counts, log color)')
plt.tight_layout(); plt.savefig(str(FIGDIR / 'project_metrics_fig1_confusion_raw.png'), dpi=150, bbox_inches='tight'); plt.show()
print('Figure 1 saved.')

Figure 1 saved.


C:\Users\user\AppData\Local\Temp\ipykernel_43544\1987682439.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.savefig(str(FIGDIR / 'project_metrics_fig1_confusion_raw.png'), dpi=150, bbox_inches='tight'); plt.show()


### Figure 2 — Row-normalized pixel confusion matrices

In [9]:
fig, axes = plt.subplots(1, len(arms), figsize=(4.2 * len(arms), 4.2))
if len(arms) == 1: axes = [axes]
for ax, a in zip(axes, arms):
    m = np.array(conf[a]['matrix_row_normalized'], dtype=float)
    im = ax.imshow(m, cmap='Blues', vmin=0, vmax=1)
    for i in range(2):
        for j in range(2):
            ax.text(j, i, f'{m[i, j]:.4f}', ha='center', va='center', fontsize=11,
                    color='black' if m[i, j] < 0.6 else 'white')
    ax.set_xticks([0, 1]); ax.set_xticklabels(['Pred BG', 'Pred FG'])
    ax.set_yticks([0, 1]); ax.set_yticklabels(['GT BG', 'GT FG'])
    ax.set_title(f'{a}\n(row-normalized)')
plt.tight_layout(); plt.savefig(str(FIGDIR / 'project_metrics_fig2_confusion_normalized.png'), dpi=150, bbox_inches='tight'); plt.show()
print('Figure 2 saved.')

Figure 2 saved.


C:\Users\user\AppData\Local\Temp\ipykernel_43544\2318888255.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.savefig(str(FIGDIR / 'project_metrics_fig2_confusion_normalized.png'), dpi=150, bbox_inches='tight'); plt.show()


## 8. Detection / Localization Metrics

In [10]:
display(size_df)
print()
display(cov_df)
print()
display(prop_summary)
print()
for arm, d in curves.items():
    if d.get('pr_curve_meaningful') is False:
        print(f'{arm}: PR/FROC NOT MEANINGFUL -> {d.get("pr_curve_not_meaningful_reason")}')

,arm,size_stratum,gt_components,detected,detection_recall,metric_domain
0,GLOBAL,TINY,72,57,0.791667,COMPONENT_DETECTION
1,GLOBAL,SMALL,28,23,0.821429,COMPONENT_DETECTION
2,GLOBAL,MEDIUM,36,31,0.861111,COMPONENT_DETECTION
3,GLOBAL,LARGE,16,14,0.875000,COMPONENT_DETECTION
4,ORACLE_REGIONAL,TINY,72,72,1.000000,COMPONENT_DETECTION
5,ORACLE_REGIONAL,SMALL,28,28,1.000000,COMPONENT_DETECTION
6,ORACLE_REGIONAL,MEDIUM,36,36,1.000000,COMPONENT_DETECTION
7,ORACLE_REGIONAL,LARGE,16,16,1.000000,COMPONENT_DETECTION
8,PREDICTED_REGIONAL,TINY,72,57,0.791667,COMPONENT_DETECTION
9,PREDICTED_REGIONAL,SMALL,28,25,0.892857,COMPONENT_DETECTION


,arm,coverage50_overall,coverage90_overall,detection_overall,tiny_detection,tiny_coverage50,tiny_coverage90,metric_domain
0,GLOBAL,0.743421,0.276316,0.822368,0.791667,0.736111,0.194444,COMPONENT_DETECTION
1,ORACLE_REGIONAL,0.980263,0.414474,1.000000,1.000000,1.000000,0.375000,COMPONENT_DETECTION
2,PREDICTED_REGIONAL,0.769737,0.289474,0.875000,0.791667,0.750000,0.305556,COMPONENT_DETECTION
3,H3_DOWNSTREAM,0.743421,0.421053,0.901316,0.819444,0.722222,0.500000,COMPONENT_DETECTION


,arm,metric_domain,images,total_proposals,proposals_per_image,tp_proposals,fp_proposals,proposal_precision,background_fraction,average_precision,pr_auc_available,froc_available,proposal_iou_mean,source,caveat
0,PHASE13B_LOCALIZER,LOCALIZATION_PROPOSAL_SUPPLY,103,494.0,4.796117,122.0,372,0.246964,0.753036,0.27539148220711346,True,True,NaN,DERIVED_FROM_SAVED_ARTIFACTS,proposal_score is heavily quantized in the per...
1,H3_PROPOSAL_SUPPLY,LOCALIZATION_PROPOSAL_SUPPLY,103,NaN,4.970874,114.0,398,0.222656,0.777344,NA_DEGENERATE_SENTINEL_SCORE,False,False,0.257998,DERIVED_FROM_SAVED_ARTIFACTS,NaN
2,PHASE13B_PROPOSAL_PER_IMAGE,LOCALIZATION_PROPOSAL_SUPPLY,106,165.0,1.556604,NaN,36,NaN,NaN,NaN,False,False,NaN,DERIVED_FROM_SAVED_ARTIFACTS,NaN



H3_PROPOSAL_SUPPLY: PR/FROC NOT MEANINGFUL -> best_proposal_score uses a -1.0 sentinel for components without an overlapping proposal; the score is therefore perfectly separable by construction and any PR/FROC curve over it is degenerate. Recorded as NOT_DERIVABLE_MEANINGFULLY.


### Figure 3 — Component detection recall by size stratum

In [11]:
fig, ax = plt.subplots(figsize=(10, 5.5))
sizes = ['TINY', 'SMALL', 'MEDIUM', 'LARGE']
arms_c = sorted(size_df['arm'].unique())
x = np.arange(len(sizes)); w = 0.8 / max(1, len(arms_c))
for i, a in enumerate(arms_c):
    vals = [float(size_df[(size_df['arm'] == a) & (size_df['size_stratum'] == s)]['detection_recall'].iloc[0])
            if len(size_df[(size_df['arm'] == a) & (size_df['size_stratum'] == s)]) else 0.0 for s in sizes]
    ax.bar(x + i * w - 0.4 + w / 2, vals, w, label=a)
ax.set_xticks(x); ax.set_xticklabels(sizes)
ax.set_ylabel('component detection recall (>=1 px)'); ax.set_ylim(0, 1.05)
ax.set_title('Component detection recall by size stratum (frozen detection domain)')
ax.legend(fontsize=8)
plt.tight_layout(); plt.savefig(str(FIGDIR / 'project_metrics_fig3_component_recall_by_size.png'), dpi=150, bbox_inches='tight'); plt.show()
print('Figure 3 saved.')

Figure 3 saved.


C:\Users\user\AppData\Local\Temp\ipykernel_43544\2079313613.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.savefig(str(FIGDIR / 'project_metrics_fig3_component_recall_by_size.png'), dpi=150, bbox_inches='tight'); plt.show()


### Figure 4 — Precision-Recall curve (Phase-13B localizer proposals)

In [12]:
p13 = curves.get('PHASE13B_LOCALIZER', {})
if p13.get('pr_curve'):
    pc = p13['pr_curve']
    order = np.argsort(pc['recall'])
    rec = np.array(pc['recall'])[order]; prec = np.array(pc['precision'])[order]
    fig, ax = plt.subplots(figsize=(7, 5.5))
    ax.plot(rec, prec, marker='.', ms=3, label=f"Phase-13B (AP={p13['average_precision_descriptive']:.4f})")
    ax.axvline(x=0.0, color='gray', lw=0.8)
    ax.set_xlabel('recall (proposal-level)'); ax.set_ylabel('precision (proposal-level)')
    ax.set_title('Descriptive PR curve — Phase-13B localizer proposals\n(DESCRIPTIVE ONLY; frozen threshold not re-selected)')
    ax.set_xlim(0, 1.02); ax.set_ylim(0, 1.02); ax.legend()
    plt.tight_layout(); plt.savefig(str(FIGDIR / 'project_metrics_fig4_pr_curve_phase13b.png'), dpi=150, bbox_inches='tight'); plt.show()
    print('Figure 4 saved.')
else:
    print('PR curve not available.')

Figure 4 saved.


C:\Users\user\AppData\Local\Temp\ipykernel_43544\1619348233.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.savefig(str(FIGDIR / 'project_metrics_fig4_pr_curve_phase13b.png'), dpi=150, bbox_inches='tight'); plt.show()


### Figure 5 — FROC curve (Phase-13B localizer proposals)

In [13]:
froc = p13.get('froc_curve')
if froc:
    fig, ax = plt.subplots(figsize=(7, 5.5))
    ax.plot(froc['fp_per_image'], froc['sensitivity'], marker='.', ms=3)
    ax.set_xlabel('false positives per image'); ax.set_ylabel('sensitivity (proposal-level)')
    ax.set_title('Descriptive FROC — Phase-13B localizer proposals\n(DESCRIPTIVE ONLY; operating point not re-selected)')
    plt.tight_layout(); plt.savefig(str(FIGDIR / 'project_metrics_fig5_froc_phase13b.png'), dpi=150, bbox_inches='tight'); plt.show()
    print('Figure 5 saved.')
else:
    print('FROC not available.')

Figure 5 saved.


C:\Users\user\AppData\Local\Temp\ipykernel_43544\3078994324.py:7: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.savefig(str(FIGDIR / 'project_metrics_fig5_froc_phase13b.png'), dpi=150, bbox_inches='tight'); plt.show()


## 9. Boundary Metrics

In [14]:
bcols = ['arm', 'image_mean_hd95', 'image_median_hd95', 'hd95_ci_low', 'hd95_ci_high', 'n_valid_hd95',
         'image_mean_assd', 'assd_ci_low', 'assd_ci_high', 'n_valid_assd',
         'image_mean_surface_dice', 'surface_dice_ci_low', 'surface_dice_ci_high', 'n_valid_surface_dice']
display(summary_df[[c for c in bcols if c in summary_df.columns]])
print()
print('UNITS: pixels. pixel_spacing_available =', backfill['pixel_spacing_available'])
print('surface tolerance (px):', backfill['surface_tolerance_px'])
print('boundary crop margin (px):', backfill['boundary_crop_margin_px'])

,arm,image_mean_hd95,image_median_hd95,hd95_ci_low,hd95_ci_high,n_valid_hd95,image_mean_assd,assd_ci_low,assd_ci_high,n_valid_assd,image_mean_surface_dice,surface_dice_ci_low,surface_dice_ci_high,n_valid_surface_dice
0,GLOBAL,332.727336,22.409465,225.704822,452.259740,98,106.590591,58.132968,170.279543,98,0.350720,0.316077,0.386983,98
1,ORACLE_REGIONAL,266.780822,9.273154,171.878158,369.555062,103,47.572100,29.965567,67.143809,103,0.502613,0.459748,0.545121,103
2,PREDICTED_REGIONAL,1612.082657,1757.638471,1494.128598,1733.547716,103,438.416992,382.406374,500.174364,103,0.190033,0.166749,0.215350,103
3,H3_DOWNSTREAM,1566.038846,1621.479493,1459.498029,1676.610684,103,389.395840,345.620371,438.030249,103,0.150363,0.132516,0.168070,103



UNITS: pixels. pixel_spacing_available = False
surface tolerance (px): 2.0
boundary crop margin (px): 256


### Figure 6 — HD95 and surface Dice distributions

In [15]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
for a in sorted(perimg_df['arm'].unique()):
    v = perimg_df[perimg_df['arm'] == a]['hd95'].astype(float).dropna()
    if len(v): ax1.hist(v, bins=25, alpha=0.5, label=a)
    w = perimg_df[perimg_df['arm'] == a]['surface_dice'].astype(float).dropna()
    if len(w): ax2.hist(w, bins=25, alpha=0.5, label=a)
ax1.set_xlabel('HD95 (pixels)'); ax1.set_ylabel('images'); ax1.set_title('HD95 distribution (pixels)'); ax1.legend(fontsize=8)
ax2.set_xlabel('surface Dice @2 px'); ax2.set_ylabel('images'); ax2.set_title('Surface Dice distribution'); ax2.legend(fontsize=8)
plt.tight_layout(); plt.savefig(str(FIGDIR / 'project_metrics_fig6_boundary_distributions.png'), dpi=150, bbox_inches='tight'); plt.show()
print('Figure 6 saved.')

Figure 6 saved.


C:\Users\user\AppData\Local\Temp\ipykernel_43544\34784572.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.savefig(str(FIGDIR / 'project_metrics_fig6_boundary_distributions.png'), dpi=150, bbox_inches='tight'); plt.show()


## 10. Calibration Metrics

In [16]:
ccols = ['arm', 'brier', 'ece', 'mce', 'roc_auc', 'average_precision', 'positive_fraction']
display(summary_df[[c for c in ccols if c in summary_df.columns]])
print()
print('Calibration note:', backfill['calibration_quantization_note'])
print('Bins:', backfill['calibration_bins'])

,arm,brier,ece,mce,roc_auc,average_precision,positive_fraction
0,GLOBAL,0.000401,0.000369,0.390488,0.827385,0.579240,0.000765
1,ORACLE_REGIONAL,0.000291,0.000220,1.000000,0.972221,0.795965,0.000765
2,PREDICTED_REGIONAL,0.000693,0.000658,0.583260,0.887877,0.467578,0.000765
3,H3_DOWNSTREAM,0.000828,0.000817,0.641287,0.879590,0.409897,0.000765



Calibration note: The frozen restore path quantizes probabilities to 1/255 levels; calibration metrics are computed on those quantized maps.
Bins: 256


### Figure 7 — Reliability diagrams (pixel-level)

In [17]:
fig, axes = plt.subplots(1, len(arms), figsize=(4.2 * len(arms), 4.2))
if len(arms) == 1: axes = [axes]
for ax, a in zip(axes, arms):
    rb = conf[a]['calibration']['reliability_bins']
    n = np.array(rb['bin_count'], dtype=float)
    c = np.array([np.nan if v is None else v for v in rb['bin_confidence']], dtype=float)
    acc = np.array([np.nan if v is None else v for v in rb['bin_accuracy']], dtype=float)
    m = (n > 0) & ~np.isnan(c)
    ax.plot([0, 1], [0, 1], 'k--', lw=1)
    if m.any():
        ax.scatter(c[m], acc[m], s=np.clip(n[m] / max(1, n[m].max()) * 120, 4, 120), alpha=0.75)
    ax.set_xlabel('mean predicted probability'); ax.set_ylabel('empirical foreground fraction')
    ax.set_title(f"{a}\nBrier={conf[a]['calibration']['brier']:.5f}  ECE={conf[a]['calibration']['ece']:.5f}")
    ax.set_xlim(0, 1); ax.set_ylim(0, 1)
plt.tight_layout(); plt.savefig(str(FIGDIR / 'project_metrics_fig7_reliability.png'), dpi=150, bbox_inches='tight'); plt.show()
print('Figure 7 saved.')

Figure 7 saved.


C:\Users\user\AppData\Local\Temp\ipykernel_43544\668657221.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.savefig(str(FIGDIR / 'project_metrics_fig7_reliability.png'), dpi=150, bbox_inches='tight'); plt.show()


## 11. Uncertainty Metrics

In [18]:
print('Bootstrap configuration:', json.dumps(backfill['bootstrap']))
print()
dcols = ['arm', 'image_mean_dice', 'dice_ci_low', 'dice_ci_high', 'image_mean_recall', 'recall_ci_low', 'recall_ci_high',
         'image_mean_precision', 'precision_ci_low', 'precision_ci_high', 'image_mean_specificity', 'specificity_ci_low', 'specificity_ci_high']
display(summary_df[[c for c in dcols if c in summary_df.columns]])
print()
display(paired_df)
print()
print('All paired statistics are labelled:', backfill['metric_labels']['paired_statistics'])

Bootstrap configuration: {"n_boot": 2000, "seed": 42, "ci_method": "percentile", "resampling_unit": "image (per-image metric); component tables use component rows"}



,arm,image_mean_dice,dice_ci_low,dice_ci_high,image_mean_recall,recall_ci_low,recall_ci_high,image_mean_precision,precision_ci_low,precision_ci_high,image_mean_specificity,specificity_ci_low,specificity_ci_high
0,GLOBAL,0.635917,0.581571,0.686591,0.657940,0.597209,0.716797,0.727183,0.676754,0.775029,0.999881,0.999845,0.999912
1,ORACLE_REGIONAL,0.802416,0.776504,0.825454,0.846539,0.821827,0.867779,0.798368,0.764316,0.829816,0.999880,0.999840,0.999914
2,PREDICTED_REGIONAL,0.418531,0.373291,0.463137,0.688882,0.634570,0.740778,0.351810,0.303980,0.401228,0.999506,0.999439,0.999571
3,H3_DOWNSTREAM,0.388628,0.346645,0.429068,0.703337,0.646026,0.757410,0.312858,0.270561,0.357565,0.999357,0.999264,0.999443


,comparison,metric,n_pairs,mean_difference,median_difference,ci_low,ci_high,wilcoxon_statistic,wilcoxon_p,label,metric_domain
0,H3_DOWNSTREAM - GLOBAL,dice,103,-0.247290,-0.241442,-0.292163,-0.200407,423.0,1.182160e-13,DESCRIPTIVE_POST_HOC_STATISTICAL_SUPPORT,PIXEL_SEGMENTATION
1,PREDICTED_REGIONAL - GLOBAL,dice,103,-0.217386,-0.206171,-0.258527,-0.175111,372.0,3.285292e-14,DESCRIPTIVE_POST_HOC_STATISTICAL_SUPPORT,PIXEL_SEGMENTATION
2,H3_DOWNSTREAM - PREDICTED_REGIONAL,dice,103,-0.029903,-0.028921,-0.051138,-0.008397,1771.0,2.845474e-03,DESCRIPTIVE_POST_HOC_STATISTICAL_SUPPORT,PIXEL_SEGMENTATION
3,ORACLE_REGIONAL - GLOBAL,dice,103,0.166499,0.074027,0.124279,0.213573,111.0,3.034700e-17,DESCRIPTIVE_POST_HOC_STATISTICAL_SUPPORT,PIXEL_SEGMENTATION



All paired statistics are labelled: DESCRIPTIVE_POST_HOC_STATISTICAL_SUPPORT


### Figure 8 — Bootstrap 95% CI forest plot (Dice)

In [19]:
fig, ax = plt.subplots(figsize=(8, 4.5))
y = np.arange(len(summary_df))
ax.errorbar(summary_df['image_mean_dice'], y,
            xerr=[summary_df['image_mean_dice'] - summary_df['dice_ci_low'],
                  summary_df['dice_ci_high'] - summary_df['image_mean_dice']],
            fmt='o', capsize=5, color='#e67e22')
ax.set_yticks(y); ax.set_yticklabels(summary_df['arm'])
ax.set_xlabel('mean per-image Dice (95% bootstrap CI)')
ax.set_title('Descriptive uncertainty — mean per-image Dice by arm')
plt.tight_layout(); plt.savefig(str(FIGDIR / 'project_metrics_fig8_dice_forest.png'), dpi=150, bbox_inches='tight'); plt.show()
print('Figure 8 saved.')

Figure 8 saved.


C:\Users\user\AppData\Local\Temp\ipykernel_43544\716078040.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.savefig(str(FIGDIR / 'project_metrics_fig8_dice_forest.png'), dpi=150, bbox_inches='tight'); plt.show()


### Figure 9 — Paired per-image Dice difference (H3 - GLOBAL)

In [20]:
piv = perimg_df.pivot_table(index='sample_id', columns='arm', values='dice')
if 'H3_DOWNSTREAM' in piv and 'GLOBAL' in piv:
    d = (piv['H3_DOWNSTREAM'] - piv['GLOBAL']).astype(float)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
    ax1.hist(d, bins=30, color='#2980b9', edgecolor='black', linewidth=0.4)
    ax1.axvline(0, color='black', lw=1)
    ax1.axvline(float(d.mean()), color='red', linestyle='--', label=f'mean={d.mean():.4f}')
    ax1.set_xlabel('per-image Dice(H3) - Dice(GLOBAL)'); ax1.set_ylabel('images')
    ax1.set_title('Paired Dice difference'); ax1.legend()
    ax2.scatter(piv['GLOBAL'], piv['H3_DOWNSTREAM'], s=12, alpha=0.6)
    lim = [0, 1]; ax2.plot(lim, lim, 'k--', lw=1)
    ax2.set_xlabel('Dice GLOBAL'); ax2.set_ylabel('Dice H3 downstream'); ax2.set_title('Paired scatter')
    plt.tight_layout(); plt.savefig(str(FIGDIR / 'project_metrics_fig9_paired_dice_diff.png'), dpi=150, bbox_inches='tight'); plt.show()
    print('Figure 9 saved.')
else:
    print('Paired data unavailable.')

Figure 9 saved.


C:\Users\user\AppData\Local\Temp\ipykernel_43544\555314979.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.savefig(str(FIGDIR / 'project_metrics_fig9_paired_dice_diff.png'), dpi=150, bbox_inches='tight'); plt.show()


## 12. H3 Descriptive Augmentation

In [21]:
h3 = summary_df[summary_df['arm'] == 'H3_DOWNSTREAM'].iloc[0]
h3_ref = load_json('reports/audits/h3_downstream_evaluation.json')
print('FROZEN H3 FORMAL METRIC (unchanged):')
print('  TinyRecall_H3 =', h3_ref['formal_downstream']['TinyRecall_H3'], '=', h3_ref['formal_downstream']['TinyRecall_H3_fraction'])
print('  R_H3          =', h3_ref['formal_downstream']['R_H3'], '->', h3_ref['formal_downstream']['interpretation_band'])
print()
print('NEW DESCRIPTIVE H3 METRICS (label:', backfill['metric_labels']['h3_specific'], ')')
print('  pooled Dice           :', h3['pooled_dice'])
print('  pooled IoU            :', h3['pooled_iou'])
print('  pooled precision      :', h3['pooled_precision'])
print('  pooled recall         :', h3['pooled_recall'])
print('  pooled specificity    :', h3['pooled_specificity'])
print('  pooled MCC            :', h3['pooled_mcc'])
print('  pooled balanced acc   :', h3['pooled_balanced_accuracy'])
print('  HD95 (px) mean        :', h3.get('image_mean_hd95'))
print('  ASSD (px) mean        :', h3.get('image_mean_assd'))
print('  surface Dice @2px     :', h3.get('image_mean_surface_dice'))
print('  Brier / ECE / ROC-AUC :', h3['brier'], '/', h3['ece'], '/', h3['roc_auc'])
print('  component TP/FP/FN    :', h3['component_tp_total'], '/', h3['component_fp_total'], '/', h3['component_fn_total'])
print('  component precision   :', h3['component_precision_overall'])
print('  FP components / image :', h3['mean_fp_components_per_image'])
print('  Dice 95% CI           :', h3['dice_ci_low'], '-', h3['dice_ci_high'])

FROZEN H3 FORMAL METRIC (unchanged):
  TinyRecall_H3 = 0.8194444444444444 = 59/72
  R_H3          = 0.13333333333333336 -> insufficient

NEW DESCRIPTIVE H3 METRICS (label: DESCRIPTIVE_SUPPORTING_METRIC__NOT_H3_GATE )
  pooled Dice           : 0.4647859863522062
  pooled IoU            : 0.3027499633408657
  pooled precision      : 0.3990265260717303
  pooled recall         : 0.5564964454577817
  pooled specificity    : 0.9993583720237394
  pooled MCC            : 0.4707568760511671
  pooled balanced acc   : 0.7779274087407606
  HD95 (px) mean        : 1566.0388459936885
  ASSD (px) mean        : 389.3958400198681
  surface Dice @2px     : 0.150362636080047
  Brier / ECE / ROC-AUC : 0.0008284817042316 / 0.0008170581805314 / 0.8795900973417459
  component TP/FP/FN    : 137 / 780 / 15
  component precision   : 0.1823899371069182
  FP components / image : 7.572815533980583
  Dice 95% CI           : 0.3466451349705586 - 0.429067919704549


## 13. Phase-14 Descriptive Augmentation

In [22]:
p14 = summary_df[summary_df['arm'].isin(['GLOBAL', 'ORACLE_REGIONAL', 'PREDICTED_REGIONAL'])]
cols14 = ['arm', 'pooled_dice', 'pooled_iou', 'pooled_precision', 'pooled_recall', 'pooled_specificity',
          'pooled_mcc', 'image_mean_hd95', 'image_mean_surface_dice', 'brier', 'ece', 'roc_auc']
display(p14[[c for c in cols14 if c in p14.columns]])
print()
print('Phase-14 frozen references reproduced exactly by the replay:')
for arm, r in backfill['reproduction_gate'].items():
    print(f"  {arm}: dice {r['replayed_mean_dice']:.10f} vs frozen {r['frozen_reference']:.10f} -> {r['match']}")

,arm,pooled_dice,pooled_iou,pooled_precision,pooled_recall,pooled_specificity,pooled_mcc,image_mean_hd95,image_mean_surface_dice,brier,ece,roc_auc
0,GLOBAL,0.665747,0.498966,0.790486,0.575010,0.999883,0.673986,332.727336,0.350720,0.000401,0.000369,0.827385
1,ORACLE_REGIONAL,0.751746,0.602238,0.821843,0.692666,0.999885,0.754325,266.780822,0.502613,0.000291,0.000220,0.972221
2,PREDICTED_REGIONAL,0.514746,0.346571,0.472663,0.565056,0.999517,0.516396,1612.082657,0.190033,0.000693,0.000658,0.887877



Phase-14 frozen references reproduced exactly by the replay:
  GLOBAL: dice 0.6359172412 vs frozen 0.6359172412 -> True
  ORACLE_REGIONAL: dice 0.8024158113 vs frozen 0.8024158113 -> True
  PREDICTED_REGIONAL: dice 0.4185309063 vs frozen 0.4185309063 -> True
  H3_DOWNSTREAM: dice 0.3886275352 vs frozen 0.3886275352 -> True


## 14. Reproducibility / Provenance

In [23]:
prov = [
    ('project_evaluation_metrics_inventory.json', 'reports/audits/project_evaluation_metrics_inventory.json'),
    ('project_evaluation_metric_policy.json', 'reports/audits/project_evaluation_metric_policy.json'),
    ('evaluation_empty_mask_policy.json', 'reports/audits/evaluation_empty_mask_policy.json'),
    ('project_metric_registry.json', 'reports/audits/project_metric_registry.json'),
    ('project_evaluation_metrics_backfill.json', 'reports/audits/project_evaluation_metrics_backfill.json'),
    ('project_evaluation_proposal_metrics_backfill.json', 'reports/audits/project_evaluation_proposal_metrics_backfill.json'),
    ('project_evaluation_segmentation_summary.csv', 'reports/tables/project_evaluation_segmentation_summary.csv'),
    ('project_evaluation_per_image_metrics.csv', 'reports/tables/project_evaluation_per_image_metrics.csv'),
    ('project_evaluation_confusion_matrices.json', 'reports/tables/project_evaluation_confusion_matrices.json'),
    ('project_evaluation_paired_comparisons.csv', 'reports/tables/project_evaluation_paired_comparisons.csv'),
    ('src/metrics/evaluation_metrics.py', 'src/metrics/evaluation_metrics.py'),
]
display(pd.DataFrame([{'artifact': n, 'path': p, 'sha256': sha(p)} for n, p in prov]))
print()
print('Safe-replay rule checks:')
for k, v in backfill['safe_replay_rule'].items():
    print(f'  {k}: {v}')

,artifact,path,sha256
0,project_evaluation_metrics_inventory.json,reports/audits/project_evaluation_metrics_inve...,af4f573578df106f89bde25f30bd95e688b337505b70d8...
1,project_evaluation_metric_policy.json,reports/audits/project_evaluation_metric_polic...,d4d3f94a61b3139747c5a0d7e061a0d8eb8083c25e688a...
2,evaluation_empty_mask_policy.json,reports/audits/evaluation_empty_mask_policy.json,16e702fd83b65e872acdab4cee06e0dd7c816f2659379d...
3,project_metric_registry.json,reports/audits/project_metric_registry.json,08647ec9d33fcfa68f74ef299a47308cbbf8d6ef3a4904...
4,project_evaluation_metrics_backfill.json,reports/audits/project_evaluation_metrics_back...,a79e531a231a03252331ff24a5bb11c2fec7d857f100c0...
5,project_evaluation_proposal_metrics_backfill.json,reports/audits/project_evaluation_proposal_met...,3f4d4d2599586185b7837ddf23779a1ad1cf5ff8ecdd71...
6,project_evaluation_segmentation_summary.csv,reports/tables/project_evaluation_segmentation...,d7cda533740600f4c2f1fd013f6f8c045317adbe140032...
7,project_evaluation_per_image_metrics.csv,reports/tables/project_evaluation_per_image_me...,4823df7e08fb87d068d2984b3170ca7cab6a6f12d04b24...
8,project_evaluation_confusion_matrices.json,reports/tables/project_evaluation_confusion_ma...,64c32e8e858eac8dd7f3688880f0dc2fd1a200287a43ec...
9,project_evaluation_paired_comparisons.csv,reports/tables/project_evaluation_paired_compa...,d21254793973b1ed18fb1b15245697a7970e02af4228de...



Safe-replay rule checks:
  1_checkpoints_frozen_and_sha_verified: True
  2_evaluator_preprocessing_frozen: True
  2_note: Reuses the frozen modules verbatim: _preprocess_image, _restore_original_coordinates, generate_predicted_rois, crop_and_preprocess, restore_regional_probability, aggregate_and_fuse.
  3_no_test_access: True
  4_no_threshold_search: True
  5_strictly_descriptive_missing_metrics: True
  6_existing_primary_metrics_reproduced_first: True
  7_does_not_change_frozen_decisions: True
  8_explicitly_logged: True
  verdict: REPLAY_PERFORMED_LAWFULLY


## 15. Future Metric Standard

In [24]:
print('Registry:', registry['artifact'], '| version', registry['version'])
print('Module  :', registry['module'], '| sha', registry['module_sha256'][:16], '...')
print()
reg_df = pd.DataFrame(registry['metrics'])
display(reg_df[['name', 'domain', 'unit', 'aggregation', 'higher_is_better', 'prospective_gate', 'descriptive_only']])
print()
print('Rule:', registry['governance']['rule'])

Registry: project_metric_registry | version 1.0.0
Module  : src/metrics/evaluation_metrics.py | sha 6c515f6f1a495315 ...



,name,domain,unit,aggregation,higher_is_better,prospective_gate,descriptive_only
0,tp_pixels,PIXEL_SEGMENTATION,pixels,sum,None,False,True
1,tn_pixels,PIXEL_SEGMENTATION,pixels,sum,None,False,True
2,fp_pixels,PIXEL_SEGMENTATION,pixels,sum,None,False,True
3,fn_pixels,PIXEL_SEGMENTATION,pixels,sum,None,False,True
4,dice,PIXEL_SEGMENTATION,ratio,mean over images AND pooled,True,True,False
5,iou,PIXEL_SEGMENTATION,ratio,mean over images AND pooled,True,False,True
6,precision,PIXEL_SEGMENTATION,ratio,mean over images AND pooled,True,False,True
7,recall,PIXEL_SEGMENTATION,ratio,mean over images AND pooled,True,True,False
8,specificity,PIXEL_SEGMENTATION,ratio,mean over images AND pooled,True,False,True
9,npv,PIXEL_SEGMENTATION,ratio,mean over images AND pooled,True,False,True



Rule: Only metrics explicitly marked prospective_gate=True may enter a frozen adjudication.


## 16. Current Governance State

In [25]:
gov = {
    'terminal_state': 'PROJECT_EVALUATION_METRICS_AUDIT_COMPLETE__H3_ADJUDICATION_PENDING',
    'h3_state': 'H3_DOWNSTREAM_EVALUATION_COMPLETE__FORMAL_ADJUDICATION_PENDING',
    'frozen_gates_changed': False,
    'checkpoints_changed': False,
    'thresholds_changed': False,
    'notebook_07_status': 'NOT_CREATED__FORMAL_ADJUDICATION_PENDING_AUTHORIZATION',
    'phase_15_status': 'NOT_STARTED_AND_NOT_AUTHORIZED',
    'test_reads': 0,
    'external_test_reads': 0,
}
for k, v in gov.items():
    print(f'{k}: {v}')

terminal_state: PROJECT_EVALUATION_METRICS_AUDIT_COMPLETE__H3_ADJUDICATION_PENDING
h3_state: H3_DOWNSTREAM_EVALUATION_COMPLETE__FORMAL_ADJUDICATION_PENDING
frozen_gates_changed: False
checkpoints_changed: False
thresholds_changed: False
notebook_07_status: NOT_CREATED__FORMAL_ADJUDICATION_PENDING_AUTHORIZATION
phase_15_status: NOT_STARTED_AND_NOT_AUTHORIZED
test_reads: 0
external_test_reads: 0


## Assertions

In [26]:
# ── Governance ───────────────────────────────────────────────────────────────
assert backfill['governance']['test_reads'] == 0
assert backfill['governance']['external_test_reads'] == 0
assert backfill['governance']['training_executed'] is False
assert backfill['governance']['thresholds_changed'] is False
assert backfill['governance']['checkpoints_changed'] is False
assert backfill['governance']['frozen_gates_changed'] is False
assert backfill['governance']['phase15_status'] == 'NOT_STARTED_AND_NOT_AUTHORIZED'
assert proposal['governance']['inference_executed'] is False
assert proposal['governance']['threshold_selected_from_curves'] is False

# ── Safe-replay conditions ───────────────────────────────────────────────────
sr = backfill['safe_replay_rule']
for k in ['1_checkpoints_frozen_and_sha_verified', '2_evaluator_preprocessing_frozen',
          '3_no_test_access', '4_no_threshold_search', '5_strictly_descriptive_missing_metrics',
          '6_existing_primary_metrics_reproduced_first', '7_does_not_change_frozen_decisions',
          '8_explicitly_logged']:
    assert sr[k] is True, k

# ── Reproduction gate: every arm reproduced its frozen primary metric ────────
for arm, r in backfill['reproduction_gate'].items():
    assert r['match'] is True, f"{arm} dice mismatch"
    assert r['tiny_match'] is True, f"{arm} tiny detection mismatch"

# ── New metrics are labelled descriptive ─────────────────────────────────────
assert backfill['metric_labels']['all_new_metrics'] == 'DESCRIPTIVE_SUPPORTING_METRIC__NOT_PROSPECTIVE_GATE'
assert backfill['metric_labels']['h3_specific'] == 'DESCRIPTIVE_SUPPORTING_METRIC__NOT_H3_GATE'

# ── H3 formal metric unchanged ───────────────────────────────────────────────
h3_ref = load_json('reports/audits/h3_downstream_evaluation.json')
assert h3_ref['formal_downstream']['TinyRecall_H3_fraction'] == '59/72'
assert abs(h3_ref['formal_downstream']['R_H3'] - 2/15) < 1e-12

# ── Table cardinality ────────────────────────────────────────────────────────
assert len(summary_df) == 4
assert len(perimg_df) == 4 * 103
assert inv['rows']
assert len(registry['metrics']) >= 30

# ── Confusion matrices are real pixel counts ─────────────────────────────────
for a, d in conf.items():
    tot = d['raw_counts']['tn'] + d['raw_counts']['fp'] + d['raw_counts']['fn'] + d['raw_counts']['tp']
    assert tot == d['pooled_pixels'] > 0

print('\n✓ All project evaluation-metrics audit assertions passed.')
print('Terminal state: PROJECT_EVALUATION_METRICS_AUDIT_COMPLETE__H3_ADJUDICATION_PENDING')


✓ All project evaluation-metrics audit assertions passed.


Terminal state: PROJECT_EVALUATION_METRICS_AUDIT_COMPLETE__H3_ADJUDICATION_PENDING
